In [1]:
import torch 
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib

SRC = pathlib.Path.cwd().parent / "moe" / "src"   # ...\LearningDeepLearning\moe\src
sys.path.insert(0, str(SRC))

from gmm_dataset import generate_dataset

In [2]:
class SimpleDNN(torch.nn.Module):
    def __init__(self, D, H, N):
        super(SimpleDNN, self).__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(D, H),
            torch.nn.ReLU(),
            torch.nn.Linear(H, H),
            torch.nn.ReLU(),
            torch.nn.Linear(H, N)
        )

    def forward(self, x):
        return self.net.forward(x)

def fit_batch(moe, train_dataloader, test_dataloader, criterion, D_out, num_epochs=2):
    optim = torch.optim.Adam(moe.parameters(), lr=1e-3)

    losses = []
    for _ in range(num_epochs):
        for (X, target_tensor) in train_dataloader:
            moe.to(X)
            optim.zero_grad()
            pred = moe.forward(X)
            loss = criterion(pred, target_tensor)
            loss.backward()
            optim.step()

            losses.append(loss.item())

    correct = 0 
    total = 0
    for (X_test, Y_test) in test_dataloader:
        preds = moe.forward(X_test)
        assert preds.shape == (X_test.shape[0], D_out), f"shape of preds {preds.shape}, is not {X_test} * {D_out}"
        preds = preds.argmax(dim=-1)
        batch_correct = (preds == Y_test).sum()
        correct += batch_correct
        total += X_test.shape[0]

    acc = correct / total 
    print(f"acc {acc}")


    plt.plot(range(len(losses)), losses)
    plt.savefig('loss_plot.png') 
    plt.close()

In [3]:
M = 10000
D = 2
N = 4
H = 1000
batch_size = 100
criterion = torch.nn.CrossEntropyLoss()
train_dataloader, test_dataloader = generate_dataset(M=M, D=D, N=N, batch_size=batch_size)
model = SimpleDNN(D, H, N)
fit_batch(model, train_dataloader, test_dataloader, criterion, N, num_epochs=100)




c:\Users\ajviswan\LearningDeepLearning\moe\src\gmm_dataset.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


acc 0.7245000004768372
